In [1]:
!pip install datasets

In [2]:
!pip install datasets==2.16.1

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
  Attempting uninstall: datasets
    Fou

##**STEP 1: Load Dataset (Task 1)**

In [ ]:
from datasets import load_dataset
dataset = load_dataset("conll2003")

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [5]:
print(dataset["train"][0])

{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [6]:
label_names = dataset["train"].features["ner_tags"].feature.names
print(label_names)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


##**STEP 2: Tokenization + Label Alignment (Task 2)**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

###Alignment Function

In [8]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["pos_tags"][word_idx])  # POS
        else:
            labels.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=False)

##**STEP 3: Model Setup (Task 3)**

In [ ]:
from transformers import AutoModelForTokenClassification

num_labels = len(dataset["train"].features["pos_tags"].feature.names)

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

##**STEP 4: Training (Task 4)**

In [15]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


##**STEP 5: Evaluation (Task 5)**

In [18]:
!pip install evaluate
!pip install seqeval
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

label_list = dataset["train"].features["pos_tags"].feature.names

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=468f27bc2d1a08e7f617bf4d59eb58e448e08b46c6cdfc44995230f63a19d890
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


###Trainer Setup + Train

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

##**STEP 6: Inference (Task 6)**

In [22]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

text = "John works at Google in California"
predictions = nlp(text)

for p in predictions:
    print(p)

{'entity': 'LABEL_22', 'score': np.float32(0.98666567), 'index': 1, 'word': 'john', 'start': 0, 'end': 4}
{'entity': 'LABEL_42', 'score': np.float32(0.4102211), 'index': 2, 'word': 'works', 'start': 5, 'end': 10}
{'entity': 'LABEL_15', 'score': np.float32(0.9417396), 'index': 3, 'word': 'at', 'start': 11, 'end': 13}
{'entity': 'LABEL_22', 'score': np.float32(0.97560716), 'index': 4, 'word': 'google', 'start': 14, 'end': 20}
{'entity': 'LABEL_15', 'score': np.float32(0.99265504), 'index': 5, 'word': 'in', 'start': 21, 'end': 23}
{'entity': 'LABEL_22', 'score': np.float32(0.9914846), 'index': 6, 'word': 'california', 'start': 24, 'end': 34}


##**STEP 7: Comparison**

###**POS Tagging**

Tags each word (Noun, Verb, Adjective)

**Example:**

John → Noun
works → Verb

**Chunking**

Groups words into phrases

**Example:**

[John] → NP
[works at Google] → VP

👉 Key Difference:

| Feature    | POS Tagging | Chunking     |
| ---------- | ----------- | ------------ |
| Level      | Word-level  | Phrase-level |
| Complexity | Easy        | Medium       |
| Output     | Labels      | Groups       |


##**STEP 8: Report Points**

**Include:**

*   **Challenges:**

    Subword token alignment

    Handling -100
*   **Observations:**
    BERT handles context well

    Chunking is harder than POS
*   **Insight:**

    Transformers outperform traditional NLP